# Local SHAP explanations

## Objective

Inspect reproducibly selected validation examples and their top raw-log-odds contributors.

## Method

Examples are selected by highest calibrated probability, distance to the frozen threshold, and median validation probability. SHAP baselines are expected raw log-odds over the training-only background.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
RESULTS = json.loads((ROOT / "reports/risk/mission03_results.json").read_text())
assert RESULTS["partitions"]["final_test_rows_loaded"] == 0
assert RESULTS["partitions"]["final_test_labels_loaded"] == 0
print("Dataset SHA-256:", RESULTS["dataset_sha256"])
print("Scope:", RESULTS["data_scope"])

## Computed results

The following cells read the generated real-data reports; they do not retrain models or load the raw final-test partition.

In [ ]:
examples = json.loads(
    (ROOT / "reports/explainability/selected_local_explanations.json").read_text()
)
[
    (
        item["selection_rule"],
        item["fraud_probability"],
        item["decision_threshold"],
        item["decision_status"],
        item["narrative"],
    )
    for item in examples
]

In [ ]:
for item in examples:
    print(item["selection_rule"], "validation source row", item["transaction_id"])
    display(pd.DataFrame(item["explanation"]["feature_contributions"]).head(10))
    print("\n".join(item["narrative"]))
display(Image(filename=str(ROOT / "reports/explainability/figures/local_shap_waterfall.png")))
display(Image(filename=str(ROOT / "reports/explainability/figures/local_linear_contributions.png")))

## Interpretation

Results are scoped to the frozen Mission 02 development partitions.

## Limitations

These examples are selected from validation, not the consumed final-test period. Contributions are not calibrated-probability deltas, causal statements, or proof of fraud.

## Conclusion

This notebook presents reproducible Mission 03 evidence while preserving the separation between predictions, calibration, risk tiers, anomalies, and explanations.